# IMDB Sentiment Classification

This is a **standalone notebook** — run it directly without needing the `.py` files.
All model definitions and training logic are inlined below.

autoresearch uses the `.py` scripts (`model.py`, `train.py`); this notebook is for
interactive exploration and prototyping.

In [ ]:
import json
import os
import time

import torch
import torch.nn as nn
import torch.optim as optim

## Model Definition

Bidirectional LSTM for binary sentiment classification.

In [ ]:
class LSTMClassifier(nn.Module):
    """Bidirectional LSTM for binary sentiment classification."""

    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, bidirectional=True, batch_first=True,
        )
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        emb = self.embedding(x)  # (batch, seq_len, embed_dim)
        _, (hidden, _) = self.lstm(emb)  # hidden: (2, batch, hidden_dim)
        # Concatenate forward and backward last hidden states
        hidden = torch.cat((hidden[0], hidden[1]), dim=1)  # (batch, hidden_dim*2)
        out = self.fc(hidden)
        return out


def make_model(vocab_size, num_classes=2):
    return LSTMClassifier(vocab_size=vocab_size, num_classes=num_classes)

## Config & Device

In [ ]:
TIME_BUDGET = int(os.environ.get("AUTORESEARCH_TIME_BUDGET", 120))
BATCH_SIZE = 64
LR = 1e-3
NUM_WORKERS = 2
DATA_DIR = os.path.join("data", "imdb")

if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

## Load Data

In [ ]:
train_data = torch.load(os.path.join(DATA_DIR, "train.pt"), weights_only=True)
val_data = torch.load(os.path.join(DATA_DIR, "val.pt"), weights_only=True)
vocab_info = torch.load(os.path.join(DATA_DIR, "vocab.pt"), weights_only=False)

vocab_size = vocab_info["vocab_size"]

train_dataset = torch.utils.data.TensorDataset(train_data["x"], train_data["y"])
val_dataset = torch.utils.data.TensorDataset(val_data["x"], val_data["y"])

trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
valloader = torch.utils.data.DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=NUM_WORKERS)

print(f"Train: {len(train_dataset)} samples, Val: {len(val_dataset)} samples")
print(f"Vocab size: {vocab_size}")

## Train

In [ ]:
model = make_model(vocab_size=vocab_size, num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params / 1e6:.2f}M")

t_start = time.time()
epoch = 0
best_val_acc = 0.0

while True:
    elapsed = time.time() - t_start
    if elapsed >= TIME_BUDGET:
        break

    epoch += 1
    model.train()
    correct = 0
    total = 0

    for inputs, targets in trainloader:
        if time.time() - t_start >= TIME_BUDGET:
            break
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    train_acc = correct / total if total > 0 else 0.0

    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for inputs, targets in valloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            val_total += targets.size(0)
            val_correct += predicted.eq(targets).sum().item()

    val_acc = val_correct / val_total
    best_val_acc = max(best_val_acc, val_acc)
    print(f"Epoch {epoch}: train_acc={train_acc:.4f} val_acc={val_acc:.4f}")

## Results

In [ ]:
total_time = time.time() - t_start

if device.type == "cuda":
    peak_memory_mb = torch.cuda.max_memory_allocated() / 1024 / 1024
elif device.type == "mps":
    peak_memory_mb = torch.mps.driver_allocated_size() / 1024 / 1024
else:
    peak_memory_mb = 0.0

metrics = {
    "val_accuracy": float(best_val_acc),
    "training_seconds": round(total_time, 1),
    "peak_memory_mb": round(peak_memory_mb, 1),
    "num_epochs": epoch,
    "num_params_M": round(num_params / 1e6, 2),
}

with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Best val_accuracy: {best_val_acc:.6f}")
print(f"Training time: {total_time:.1f}s")
print(f"Metrics written to metrics.json")